In [0]:
from pyspark.sql.functions import *
from pyspark.sql.functions import col, when, lit, rand, floor, concat, length, trim, to_date
from pyspark.sql.functions import current_date, col
from pyspark.sql.types import StringType
from pyspark.sql.functions import col, lower, trim
from pyspark.sql import Window
from pyspark.sql.types import DateType

In [0]:
# this is only for basic cleaning
class BaseCleaning:
    try:
        def __init__(self,df):
            self.df = df
        def lower_trim(self):
            for fields in self.df.schema:
                if isinstance(fields.dataType, StringType):
                    self.df = self.df.withColumn(fields.name,lower(trim(col(fields.name))))
            print("lower and trim is done")
    except Exception as e:
        print("you have erroe in base cleaning",e)

In [0]:
class Coustomer_info(BaseCleaning):
    try:
        def Coustmer_changes(self):
            renames = {
                            "cst_id":"coustmer_id",
                            "cst_key":"coustmer_key",
                            "cst_firstname":"coustmer_firstname",
                            "cst_lastname":"coustmer_lastname",
                            "cst_marital_status":"marriage_status",
                            "cst_gndr":"coustmer_gender",
                            "cst_create_date":"coustmer_join_date"
                }
            for old,new in renames.items():
                self.df = self.df.withColumnRenamed(old,new)
            #filtering Null values
            self.df = self.df.filter(col("coustmer_id").isNotNull())
            self.df = self.df.dropDuplicates(["coustmer_id"])
            print("checked_null values and dropedduplicates")
            #normilaation
            self.df = self.df.withColumn(
                                            "marriage_status",
                                                    when(upper(trim(col("marriage_status"))) == "S","single")
                                                    .when(upper(trim(col("marriage_status"))) == "M","married")
                                                    .otherwise("n/a")
            )
            print("marriage_status normilization done")
            self.df = self.df.withColumn( "coustmer_gender",
                                                when(trim(col("coustmer_gender")) == "m","male")
                                                .when(trim(col("coustmer_gender")) == "f","female")
                                                .otherwise("n/a")
            )
            print("coustmer_gender")
            self.df = self.df.withColumn("data_added",current_date())
            print("sending the data to silver.cleaned_coustmer")
            self.df =  self.df.write.mode("overwrite").format("delta").saveAsTable("lakehouse.sliver.cleaned_coustmer")
            return self.df
    except Exception as e:
        print("you have error in coustmer_change",e)

In [0]:
 class Product_info(BaseCleaning):
    def Product_Cleaning(self):
        try:
            RENAME_MAP = {
            "prd_id": "product_id",
            "cat_id": "category_id",
            "prd_key": "product_number",
            "prd_nm": "product_name",
            "prd_cost": "product_cost",
            "prd_line": "product_line",
            "prd_start_dt": "start_date",
            "sls_prd_key":"sales_product_key"

            }
            #extracting data to match with sales data and px_cat,
            self.df = self.df.withColumn("cat_id",regexp_replace(substring(col("prd_key"),1,5),"-","_"))
            self.df = self.df.withColumn("sls_prd_key",substring(col("prd_key"),7,length(col("prd_key"))))
            print("done regex")
            self.df = self.df.withColumn("prd_cost",coalesce(col("prd_cost"),lit(0)))
            print("cost 0")
            self.df = self.df.withColumn("prd_line",
                                    when(upper(col("prd_line")) == "M","Mountain")
                                    .when(trim(upper(col("prd_line"))) == "R","Road")
                                    .when(upper(col("prd_line")) == "S","Other Sales")
                                    .when(upper(col("prd_line")) == "T","Touring")
                                    .otherwise("n/a")
            )
            print("normalization")
            windows = Window.partitionBy("prd_start_dt").orderBy("prd_start_dt")
            #adjusting the end date 
            self.df = self.df.withColumn("end_date", date_sub(lead("prd_start_dt").over(windows), 1))
            print("date has been substracted")
            # changing into date
            for fields in self.df.schema:
                if isinstance(fields.dataType, DateType):
                    self.df = self.df.withColumn(fields.name, to_date(col(fields.name), "yyyy-MM-dd"))
            print("converted the date")
            for old,new in RENAME_MAP.items():
                self.df = self.df.withColumnRenamed(old,new)
            print("renamedcolumns")
            self.df = self.df.withColumn("data_added",current_date())
            print("sending the data into silver layer")
            self.df.write.format("delta").option("mode","overwrite").saveAsTable("lakehouse.sliver.crm_product")
        except Exception as e:
            print("you have error at product_info_class",e)
            

In [0]:
class Sales_Details(BaseCleaning):
    def Sales_Cleaning(self):
        try:
            RENAME_MAP = {
                            "sls_ord_num": "order_number",
                            "sls_prd_key": "product_number",
                            "sls_cust_id": "customer_id",
                            "sls_order_dt": "order_date",
                            "sls_ship_dt": "ship_date",
                            "sls_due_dt": "due_date",
                            "sls_sales": "sales_amount",
                            "sls_quantity": "quantity",
                            "sls_price": "price"
                        }
            #changing the data type to date type
            new = ['sls_order_dt','sls_ship_dt','sls_due_dt']
            for i in new:
                self.df = self.df.withColumn(i,when(
                                                    (col(i) == 0) | (length(trim(col(i).cast("string"))) != 8),None
                                                ).otherwise(to_date(trim(col(i).cast("string")), "yyyyMMdd")))
            print("date format has been changed")
            #sales and price correction
            self.df = self.df.withColumn("sls_price",
                                        when((col("sls_price").isNull()) | (col("sls_price") <= 0),
                                            when(col("sls_quantity") != 0,
                                                 abs(col("sls_sales")) / col("sls_quantity")
                                                ).otherwise(None)
                                        ).otherwise(col("sls_price"))
                                        )
            print("sales and price correction done")
            self.df = self.df.withColumn("sls_sales", when(
                                        (col("sls_sales").isNull()) | (col("sls_sales") <= 0),
                                            col("sls_price") * col("sls_quantity")
                                            ).otherwise(col("sls_sales"))
                                        )
            print("done with sls_sales")
            #changing the column names
            for old,new in RENAME_MAP.items():
                self.df = self.df.withColumnRenamed(old,new)
            print("changed the  column names")
            #adding the current date
            self.df = self.df.withColumn("date_added",current_date())
            print("wirting down to the silver layer")
            self.df.write.mode("overwrite").format("delta").saveAsTable("lakehouse.sliver.sales_details")
        except Exception as e:
            print("you have error in Sales Details",e)

In [0]:
#main class
bronze_db = "lakehouse.bronze"
tables = spark.catalog.listTables(bronze_db)
for i in tables:
    df = spark.read.table(f"{bronze_db}.{i.name}")
    if i.name == "cust_info":
        obj = Coustomer_info(df)
        obj.lower_trim()
        new_obj = obj.Coustmer_changes()
        #new_obj.show()
    elif i.name == "prd_info":
        obj1 = Product_info(df)
        obj1.lower_trim()
        new_obj1 = obj1.Product_Cleaning()
    elif i.name == "sales_details":
        obj2 = Sales_Details(df)
        obj2.lower_trim()
        obj2.Sales_Cleaning()

In [0]:
sales = spark.read.table("lakehouse.sliver.sales_details")#.show()
sales.filter(col("sales_amount").isNull()).show()

In [0]:
%sql
drop table if exists lakehouse.sliver.cleaned_coustmer;